In [1]:
import pandas as pd

states = pd.read_csv("../data/cleaned/states_combined.csv")
poll = pd.read_csv("../data/cleaned/2027_poll_by_zone_clean.csv")

cols = ["tinubu_votes", "atiku_votes", "obi_votes", "total_valid_votes"]
zone23 = states.groupby("zone")[cols].sum()

for c in ["tinubu", "atiku", "obi"]:
    zone23[f"{c}_2023_pct"] = (zone23[f"{c}_votes"] / zone23["total_valid_votes"] * 100).round(2)

zone23[["tinubu_2023_pct", "atiku_2023_pct", "obi_2023_pct"]]

,tinubu_2023_pct,atiku_2023_pct,obi_2023_pct
zone,,,
North Central,38.58,25.46,31.01
North East,34.48,50.61,9.17
North West,39.64,34.82,5.23
South East,5.72,4.09,87.78
South South,27.99,25.12,42.37
South West,53.59,22.15,19.90


In [2]:
poll = poll.set_index("zone")
for c in ["tinubu", "obi", "atiku"]:
    zone23[f"{c}_swing"] = (poll[f"{c}_pct"] - zone23[f"{c}_2023_pct"]).round(2)
zone23[["tinubu_swing", "obi_swing", "atiku_swing"]]

,tinubu_swing,obi_swing,atiku_swing
zone,,,
North Central,-1.61,0.13,NaN
North East,-11.00,11.13,4.85
North West,3.46,11.61,4.59
South East,1.94,-23.81,24.05
South South,3.00,-0.89,-0.24
South West,-10.06,11.01,NaN


In [3]:
weight = 0.5
swing = zone23[["tinubu_swing", "obi_swing", "atiku_swing"]].fillna(0)
est = states.merge(swing, left_on="zone", right_index=True)

for c in ["tinubu", "obi", "atiku"]:
    est[f"{c}_2027_pct"] = (est[f"{c}_pct"] + weight * est[f"{c}_swing"]).clip(lower=0).round(2)

est[["state", "zone", "tinubu_2027_pct", "obi_2027_pct", "atiku_2027_pct"]]

,state,zone,tinubu_2027_pct,obi_2027_pct,atiku_2027_pct
0,Abia,South East,3.38,76.50,18.16
1,Adamawa,North East,19.51,20.02,59.54
2,Akwa Ibom,South South,30.44,23.46,38.43
3,Anambra,South East,1.80,83.34,13.50
4,Bauchi,North East,31.60,8.78,52.40
5,Bayelsa,South South,27.25,29.78,41.50
6,Benue,North Central,39.52,40.10,16.89
7,Borno,North East,48.72,7.12,43.46
8,Cross River,South South,32.80,42.70,22.77
9,Delta,South South,16.16,55.10,26.14


In [4]:
est.to_csv("../data/final/state_estimates_2027.csv", index=False)